# Pleiotropic vectors

Each allele has an effect size and a continuous direction in trait space (`?EvoMod.PleiotropicVectors`). Compares corridor and divergent directional selection and tracks the classification of effect vectors (Local, Modular, Antagonistic, Integrated, Intra module).

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using EvoMod, EvoMod.PleiotropicVectors
using Gadfly, DataFrames, LinearAlgebra, Random, Statistics, DelimitedFiles
include("plots.jl")
Random.seed!(42)

In [ ]:
# Parameters
n_e    = 5000
m      = 200
p      = 4
mu     = 0.001   # mutation rate per allele (hits size and direction)
mu_σ   = 0.1     # standard deviation of mutations in effect size
mu_σ_b = 0.1     # standard deviation of mutations in effect direction

omega_var = 5
delta_theta_speed = 40/1000
theta_0 = 10 * ones(p)

regimes     = ["Drift", "Stab", "Direct"]
generations = [5000, 5000, 5000]
thin        = 100

random_ind = Individual(m, p, mu, mu_σ, mu_σ_b)

delta_theta   = delta_theta_speed * [ones(p ÷ 2); -ones(p ÷ 2)]    # divergent
delta_theta_1 = delta_theta_speed * [ones(p ÷ 2); zeros(p ÷ 2)]    # corridor
omega = twoModuleMatrix(omega_var * ones(p), 0.8, 0.8)

In [ ]:
corridor_pop = run_pop(random_ind, n_e, regimes, generations;
                       theta = theta_0, delta_theta = delta_theta_1, omega = omega, thin = thin)
divergent_pop = run_pop(random_ind, n_e, regimes, generations;
                        theta = theta_0, delta_theta = delta_theta, omega = omega, thin = thin)

In [ ]:
all_plots(corridor_pop, "pleioVectors_corridor.pdf")
all_plots(divergent_pop, "pleioVectors_divergent.pdf")

Effect-vector classes at the end of stabilizing selection and at the end of each directional regime:

In [ ]:
stab_end = sum(generations[1:2]) ÷ thin   # last moment recorded under stabilizing selection
D = vcat(corridor_pop.moments[end]["count_b"],
         corridor_pop.moments[stab_end]["count_b"],
         divergent_pop.moments[end]["count_b"])
D.regime = repeat(["Corridor", "Stabilizing", "Divergent"], inner = nrow(D) ÷ 3)
plot(D, x = :regime, y = :count, color = :class, Geom.bar(position = :stack),
     Guide.xlabel("Selection regime"), Guide.ylabel("Genetic vector count"),
     Guide.colorkey(title = ""))

In [ ]:
mkpath("data")
writedlm("data/corridor.txt", corridor_pop.moments[end]["corrG"])
writedlm("data/divergent.txt", divergent_pop.moments[end]["corrG"])
writedlm("data/stabilizing.txt", divergent_pop.moments[stab_end]["corrG"])
divergent_pop.moments[end]["corrG"]

In [ ]:
plotCorrelations(corridor_pop)

In [ ]:
plotTraits(corridor_pop)

In [ ]:
plotPleiotropy(corridor_pop)